# 🌱 AgriSense AI — Module 5: Conditional VAE for Rare Leaf Disease Augmentation
### Owner: F (Generator) | Partner: G (Augmentation & Evaluation)
**Target:** Generate synthetic images for under-represented PlantVillage disease classes to fix class imbalance for MobileNetV2 retrain.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### What this notebook produces (Deliverables for G):
1. `synthetic/<class_folder_name>/*.png` — 300 to 500 synthetic images per rare class (224x224 RGB PNG)
2. `cvae.pt` — Trained Conditional VAE weights
3. `samples/grid_<class>.png` — 8x8 visual quality grids
4. `metrics.json` — Reconstruction loss, KL divergence, and nearest-neighbor distance metrics
5. `CLASSES.txt` — List of chosen classes and sample counts
6. `synthetic_dataset_for_G.zip` — Single compressed zip file for 1-click delivery to G


In [ ]:
# Step 1: Environment Setup & GPU Verification
import os
import sys
import json
import math
import time
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.utils import make_grid, save_image

# Set reproducible seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Running on device: {device}")
if device.type == 'cuda':
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"[*] Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
else:
    print("[!] GPU not detected. Running on CPU (Colab: Runtime -> Change runtime type -> T4 GPU recommended).")


## 2. Target Class Configuration & Contract
As identified in `idea.md` and `TODO_F.md`, we pick the 4–5 rarest classes in the PlantVillage color dataset.
The Conditional VAE trains on all 5 classes simultaneously using class-embedding conditioning.


In [ ]:
# Configuration Parameters
TARGET_CLASSES = [
    "Potato___healthy",
    "Apple___Cedar_apple_rust",
    "Peach___healthy",
    "Raspberry___healthy",
    "Tomato___Tomato_mosaic_virus"
]

NUM_CLASSES = len(TARGET_CLASSES)
class_to_idx = {cls: idx for idx, cls in enumerate(TARGET_CLASSES)}
idx_to_class = {idx: cls for idx, cls in enumerate(TARGET_CLASSES)}

# Hyperparameters
IMG_SIZE = 64              # Training resolution (stable and fast)
UPSCALE_SIZE = 224         # Output resolution required by G for MobileNetV2
LATENT_DIM = 128           # Latent vector dimension z
EMBED_DIM = 32             # Class conditioning embedding dimension
BATCH_SIZE = 64            # Batch size
LEARNING_RATE = 1e-3       # Adam optimizer learning rate
EPOCHS = 60                # 60 epochs takes ~2-3 mins on Colab GPU
NUM_SYNTHETIC_PER_CLASS = 400  # Target synthetic images per class (default 400)

print(f"[*] Configured {NUM_CLASSES} rare target classes:")
for idx, cls in idx_to_class.items():
    print(f"    [{idx}] {cls}")


## 3. Dataset Pipeline
We load the real training images for the target classes.
- **Rule 1:** Only `split == 'train'` is used (never test/val).
- **Rule 2:** Normalized to `[-1, 1]` with `Tanh` output from the decoder.
- **Note:** If `splits.csv` or PlantVillage is not yet uploaded, this script includes a sample generator so you can test the entire pipeline end-to-end immediately.


In [ ]:
# Custom Dataset for Rare Leaf Classes
class RareLeafDataset(Dataset):
    def __init__(self, samples_list, transform=None):
        """
        samples_list: list of tuples (image_path, class_idx)
        """
        self.samples = samples_list
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# Transform: 64x64, horizontal flip, normalized to [-1, 1]
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Helper function to discover dataset files
def build_dataset_samples(dataset_dir="plantvillage", splits_csv="splits.csv"):
    samples = []
    
    # Priority 1: Check splits.csv from G
    if os.path.exists(splits_csv):
        print(f"[*] Found splits file: {splits_csv}")
        df = pd.read_csv(splits_csv)
        train_df = df[df['split'] == 'train']
        for _, row in train_df.iterrows():
            cls_name = row['class']
            if cls_name in class_to_idx and os.path.exists(row['path']):
                samples.append((row['path'], class_to_idx[cls_name]))
                
    # Priority 2: Scan directory directly
    if not samples and os.path.exists(dataset_dir):
        print(f"[*] Scanning directory: {dataset_dir}")
        for cls_name, cls_idx in class_to_idx.items():
            folder = Path(dataset_dir) / cls_name
            if folder.exists():
                for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
                    for img_p in folder.glob(ext):
                        samples.append((str(img_p), cls_idx))
                        
    # Priority 3: Fallback demo mock images for zero-friction dry run
    if not samples:
        print("[!] No local dataset found. Creating realistic demo sample images for pipeline testing...")
        demo_dir = Path("demo_plantvillage")
        demo_dir.mkdir(parents=True, exist_ok=True)
        colors = [(34, 139, 34), (139, 69, 19), (60, 179, 113), (46, 139, 87), (205, 92, 92)]
        for cls_idx, (cls_name, col) in enumerate(zip(TARGET_CLASSES, colors)):
            cls_dir = demo_dir / cls_name
            cls_dir.mkdir(parents=True, exist_ok=True)
            for i in range(120): # 120 samples per class
                img = Image.new('RGB', (128, 128), color=(220, 220, 220))
                # Draw simple leaf-like ellipse with noise
                from PIL import ImageDraw
                draw = ImageDraw.Draw(img)
                box = [20 + random.randint(-5, 5), 20 + random.randint(-5, 5), 108 + random.randint(-5, 5), 108 + random.randint(-5, 5)]
                draw.ellipse(box, fill=(col[0] + random.randint(-15, 15), col[1] + random.randint(-15, 15), col[2] + random.randint(-15, 15)))
                img_path = cls_dir / f"sample_{i:04d}.png"
                img.save(img_path)
                samples.append((str(img_path), cls_idx))
        print(f"[+] Created {len(samples)} demo images under {demo_dir}")

    return samples

dataset_samples = build_dataset_samples()
print(f"[*] Total training samples loaded: {len(dataset_samples)}")

# Count per class
counts = {cls: 0 for cls in TARGET_CLASSES}
for _, label in dataset_samples:
    counts[idx_to_class[label]] += 1

print("[*] Training distribution:")
for cls, count in counts.items():
    print(f"    - {cls}: {count} images")

# DataLoader
train_dataset = RareLeafDataset(dataset_samples, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True if device.type=='cuda' else False)


## 4. Conditional VAE (cVAE) Architecture
The network consists of:
- **Conditioning:** Class index $y \in \{0, \dots, 4\}$ embedded into a 32-dim vector.
- **Encoder:** 4-stage convolutional downsampling ($64 \times 64 \rightarrow 4 \times 4$), concatenated with label embedding $\rightarrow$ predicts $\mu$ and $\log(\sigma^2)$.
- **Reparameterization Trick:** $z = \mu + \epsilon \odot \exp(0.5 \cdot \log(\sigma^2))$, where $\epsilon \sim \mathcal{N}(0, I)$.
- **Decoder:** Concatenates $z$ with label embedding $\rightarrow$ 4-stage transposed convolutions $\rightarrow$ `Tanh` output in range $[-1, 1]$.


In [ ]:
# Conditional VAE Definition
class ConditionalVAE(nn.Module):
    def __init__(self, num_classes=5, latent_dim=128, embed_dim=32):
        super(ConditionalVAE, self).__init__()
        self.latent_dim = latent_dim
        self.embed_dim = embed_dim
        
        # Class embedding
        self.label_embedding = nn.Embedding(num_classes, embed_dim)
        
        # ---------------- ENCODER ----------------
        # 64x64x3 -> 32x32x64
        self.enc_conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True)
        )
        # 32x32x64 -> 16x16x128
        self.enc_conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True)
        )
        # 16x16x128 -> 8x8x256
        self.enc_conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True)
        )
        # 8x8x256 -> 4x4x512
        self.enc_conv4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # Flattened feature size = 512 * 4 * 4 = 8192
        self.enc_fc_mu = nn.Linear(512 * 4 * 4 + embed_dim, latent_dim)
        self.enc_fc_logvar = nn.Linear(512 * 4 * 4 + embed_dim, latent_dim)
        
        # ---------------- DECODER ----------------
        self.dec_input = nn.Linear(latent_dim + embed_dim, 512 * 4 * 4)
        
        # 4x4x512 -> 8x8x256
        self.dec_conv1 = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        # 8x8x256 -> 16x16x128
        self.dec_conv2 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        # 16x16x128 -> 32x32x64
        self.dec_conv3 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        # 32x32x64 -> 64x64x3 (Tanh output)
        self.dec_conv4 = nn.Sequential(
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def encode(self, x, y):
        h = self.enc_conv1(x)
        h = self.enc_conv2(h)
        h = self.enc_conv3(h)
        h = self.enc_conv4(h)
        h = torch.flatten(h, start_dim=1)  # (B, 8192)
        
        y_embed = self.label_embedding(y)  # (B, embed_dim)
        h_cond = torch.cat([h, y_embed], dim=1)
        
        mu = self.enc_fc_mu(h_cond)
        logvar = self.enc_fc_logvar(h_cond)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y):
        y_embed = self.label_embedding(y)
        z_cond = torch.cat([z, y_embed], dim=1)
        
        h = self.dec_input(z_cond)
        h = h.view(-1, 512, 4, 4)
        h = self.dec_conv1(h)
        h = self.dec_conv2(h)
        h = self.dec_conv3(h)
        out = self.dec_conv4(h)
        return out

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z, y)
        return recon_x, mu, logvar

    def sample(self, num_samples, class_idx, device, truncation=1.5):
        self.eval()
        with torch.no_grad():
            z = torch.randn(num_samples, self.latent_dim, device=device)
            if truncation > 0:
                z = torch.clamp(z, -truncation, truncation)
            y = torch.full((num_samples,), class_idx, dtype=torch.long, device=device)
            samples = self.decode(z, y)
        return samples

# Instantiate model
model = ConditionalVAE(num_classes=NUM_CLASSES, latent_dim=LATENT_DIM, embed_dim=EMBED_DIM).to(device)
print(f"[*] cVAE Model Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 5. Loss Function with $\beta$-Annealing
The objective consists of:
$$\mathcal{L} = \mathcal{L}_{\text{recon}}(x, \hat{x}) + \beta \cdot D_{\text{KL}}(q_\phi(z|x, y) \parallel p(z))$$
- **Reconstruction:** Mean Squared Error (MSE) on pixel values.
- **KL Divergence:** Analytical KL between Gaussian posterior and standard normal prior.
- **$\beta$-Annealing:** Warm up $\beta$ from $0.0$ to $0.001$ over the first 15 epochs. This ensures the model learns accurate leaf shapes and vein textures before regularizing the latent space, avoiding excessive blur.


In [ ]:
# Loss function definition
def cvae_loss_fn(recon_x, x, mu, logvar, beta=0.001):
    # Reconstruction loss (MSE)
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    
    # KL Divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = torch.mean(-0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

# Optimizer and LR Scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)


## 6. Training Execution
We train the cVAE for 60 epochs. Every 10 epochs, a sample reconstructed batch is saved to monitor visual fidelity.


In [ ]:
# Training Loop
os.makedirs("samples", exist_ok=True)
history = {'epoch': [], 'total_loss': [], 'recon_loss': [], 'kl_loss': []}

print("[*] Starting cVAE training...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_total = 0.0
    running_recon = 0.0
    running_kl = 0.0
    
    # Beta annealing schedule: warm up from 0 to 0.001 over first 15 epochs
    beta = min(0.001, 0.001 * (epoch / 15.0))
    
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        recon_x, mu, logvar = model(batch_x, batch_y)
        loss, recon_l, kl_l = cvae_loss_fn(recon_x, batch_x, mu, logvar, beta=beta)
        
        loss.backward()
        optimizer.step()
        
        running_total += loss.item() * batch_x.size(0)
        running_recon += recon_l.item() * batch_x.size(0)
        running_kl += kl_l.item() * batch_x.size(0)
        
    scheduler.step()
    
    total_l = running_total / len(train_dataset)
    recon_l = running_recon / len(train_dataset)
    kl_l = running_kl / len(train_dataset)
    
    history['epoch'].append(epoch)
    history['total_loss'].append(total_l)
    history['recon_loss'].append(recon_l)
    history['kl_loss'].append(kl_l)
    
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{EPOCHS}] - Total: {total_l:.4f} | Recon (MSE): {recon_l:.4f} | KL: {kl_l:.2f} | beta: {beta:.5f}")

total_duration = time.time() - start_time
print(f"[+] Training completed in {total_duration/60:.2f} minutes.")

# Plot training curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history['epoch'], history['recon_loss'], label='Reconstruction (MSE)', color='forestgreen')
plt.title('Reconstruction Loss')
plt.xlabel('Epoch')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['epoch'], history['kl_loss'], label='KL Divergence', color='darkorange')
plt.title('KL Divergence')
plt.xlabel('Epoch')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("samples/training_curves.png", dpi=150)
plt.show()


## 7. Deliverable 1: Visual Sample Grids (`samples/grid_<class>.png`)
We generate an 8x8 visual grid (64 samples) for each of the 5 rare classes to visually inspect realism.


In [ ]:
# Generate and display 8x8 sample grids
model.eval()

for cls_name, cls_idx in class_to_idx.items():
    samples = model.sample(num_samples=64, class_idx=cls_idx, device=device, truncation=1.5)
    # Denormalize from [-1, 1] to [0, 1]
    samples = (samples + 1.0) / 2.0
    samples = torch.clamp(samples, 0.0, 1.0)
    
    grid_path = f"samples/grid_{cls_name}.png"
    save_image(samples, grid_path, nrow=8, padding=2)
    print(f"[+] Saved 8x8 grid for '{cls_name}' -> {grid_path}")

# Display one sample grid inline
plt.figure(figsize=(8, 8))
sample_img = Image.open(f"samples/grid_{TARGET_CLASSES[0]}.png")
plt.imshow(sample_img)
plt.title(f"Sample Synthetic Grid: {TARGET_CLASSES[0]}")
plt.axis('off')
plt.show()


## 8. Deliverable 2: Batch Generation & 224x224 Upscaling (`synthetic/`)
As specified in the contract:
- Generate 300 to 500 images per class (default 400).
- Upscale from $64 \times 64 \rightarrow 224 \times 224$ via bicubic interpolation with anti-aliasing.
- Store under exact folder names: `synthetic/<exact_class_folder_name>/%04d.png`.


In [ ]:
# Batch Generation and Upscaling (64 -> 224)
output_base_dir = Path("synthetic")
output_base_dir.mkdir(parents=True, exist_ok=True)

print(f"[*] Generating {NUM_SYNTHETIC_PER_CLASS} images per class (Upscaled to {UPSCALE_SIZE}x{UPSCALE_SIZE})...")

for cls_name, cls_idx in class_to_idx.items():
    class_dir = output_base_dir / cls_name
    class_dir.mkdir(parents=True, exist_ok=True)
    
    # Generate in batches to avoid GPU OOM
    generated_count = 0
    batch_gen_size = 50
    
    while generated_count < NUM_SYNTHETIC_PER_CLASS:
        current_batch = min(batch_gen_size, NUM_SYNTHETIC_PER_CLASS - generated_count)
        
        # Sample latent vectors with truncation
        samples = model.sample(num_samples=current_batch, class_idx=cls_idx, device=device, truncation=1.5)
        
        # Denormalize [-1, 1] -> [0, 1]
        samples = (samples + 1.0) / 2.0
        samples = torch.clamp(samples, 0.0, 1.0)
        
        # Upscale 64x64 to 224x224 using bicubic interpolation
        upscaled_samples = F.interpolate(
            samples,
            size=(UPSCALE_SIZE, UPSCALE_SIZE),
            mode='bicubic',
            align_corners=False
        )
        
        # Save individual PNG images
        for i in range(current_batch):
            img_tensor = upscaled_samples[i]
            img_path = class_dir / f"{generated_count + 1:04d}.png"
            save_image(img_tensor, str(img_path))
            generated_count += 1
            
    print(f"    [+] Saved {generated_count} images -> {class_dir}")

print("[+] Synthetic dataset generation complete!")


## 9. Deliverable 3: Memorization & Novelty Check
To confirm the generator is synthesizing novel visual variations rather than merely memorizing training exemplars, we find the nearest real train image (L2 pixel distance) for synthetic samples and plot them side-by-side.


In [ ]:
# Nearest-Neighbor Memorization Test
print("[*] Running Nearest-Neighbor Memorization Check...")

# Load all real training images of class 0 to compute distance
test_cls_idx = 0
test_cls_name = TARGET_CLASSES[test_cls_idx]
real_tensors = []

for path, label in dataset_samples:
    if label == test_cls_idx:
        img = Image.open(path).convert('RGB').resize((64, 64))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        real_tensors.append(arr.flatten())

real_matrix = np.array(real_tensors) # (N_real, 64*64*3)

# Generate 5 probe synthetic images
probe_samples = model.sample(num_samples=5, class_idx=test_cls_idx, device=device, truncation=1.5)
probe_samples = (probe_samples + 1.0) / 2.0
probe_np = probe_samples.cpu().numpy().transpose(0, 2, 3, 1) # (5, 64, 64, 3)

fig, axes = plt.subplots(5, 2, figsize=(6, 12))
fig.suptitle(f"Novelty Check: Synthetic vs Nearest Real ({test_cls_name})", fontsize=12)

min_distances = []
for i in range(5):
    synth_flat = probe_np[i].flatten()
    # Compute L2 Euclidean distance to all real training images
    dists = np.linalg.norm(real_matrix - synth_flat, axis=1)
    nearest_idx = np.argmin(dists)
    min_dist = dists[nearest_idx]
    min_distances.append(float(min_dist))
    
    nearest_real = real_matrix[nearest_idx].reshape(64, 64, 3)
    
    # Plot Synthetic
    axes[i, 0].imshow(np.clip(probe_np[i], 0, 1))
    axes[i, 0].set_title(f"Synthetic #{i+1}")
    axes[i, 0].axis('off')
    
    # Plot Nearest Real
    axes[i, 1].imshow(np.clip(nearest_real, 0, 1))
    axes[i, 1].set_title(f"Nearest Real (L2: {min_dist:.2f})")
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig("samples/nearest_neighbor_check.png", dpi=150)
plt.show()

print(f"[+] Average Nearest-Neighbor L2 Distance: {np.mean(min_distances):.3f}")
print("    (Substantial distance indicates the model generates novel leaves, not 1:1 copies)")


## 10. Deliverable 4: Package Hand-off Files
We export:
- `generator.pt` (or `cvae.pt`)
- `CLASSES.txt`
- `metrics.json`
- `synthetic_dataset_for_G.zip` (ready for G to download and train MobileNetV2)


In [ ]:
# Save model weights
torch.save(model.state_dict(), "cvae.pt")
print("[+] Model weights saved to 'cvae.pt'")

# Save CLASSES.txt
with open("CLASSES.txt", "w") as f:
    for cls in TARGET_CLASSES:
        f.write(f"{cls} : {counts[cls]} real train images\n")
print("[+] Saved 'CLASSES.txt'")

# Save metrics.json
metrics_dict = {
    "model_type": "Conditional VAE",
    "target_classes": TARGET_CLASSES,
    "latent_dim": LATENT_DIM,
    "epochs_trained": EPOCHS,
    "final_reconstruction_mse": float(history['recon_loss'][-1]),
    "final_kl_divergence": float(history['kl_loss'][-1]),
    "average_nearest_neighbor_l2": float(np.mean(min_distances)),
    "synthetic_images_per_class": NUM_SYNTHETIC_PER_CLASS,
    "synthetic_image_resolution": f"{UPSCALE_SIZE}x{UPSCALE_SIZE}"
}

with open("metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=4)
print("[+] Saved 'metrics.json'")

# Compress into synthetic_dataset_for_G.zip
print("[*] Creating zip archive for delivery to G...")
zip_filename = "synthetic_dataset_for_G.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add synthetic images
    for root, dirs, files in os.walk("synthetic"):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, arcname=os.path.relpath(file_path, "."))
            
    # Add deliverables
    for extra in ["CLASSES.txt", "metrics.json", "cvae.pt", "samples/nearest_neighbor_check.png"]:
        if os.path.exists(extra):
            zipf.write(extra, arcname=extra)

print(f"[🎉] SUCCESS! Delivery package created: '{zip_filename}' ({os.path.getsize(zip_filename)/(1024*1024):.2f} MB)")
print("You can send this zip directly to Owner G or upload it to your shared Google Drive.")

# In Google Colab, trigger automatic browser download
try:
    from google.colab import files
    print("[*] Triggering download in Google Colab...")
    files.download(zip_filename)
except Exception:
    pass
